In [ ]:
import os
import pandas as pd
from datetime import datetime

data_path_specific = r'Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions'

def load_all_csvs_from_folders(root_path):
    """
    Recursively loads all CSV files from all subfolders under root_path into a single DataFrame.
    Only includes files whose name ends with today's date in YYYYMMDD format (e.g., 20250616.csv).
    Excludes folders with titles 'automation scripts', 'Logs', or 'LogsBackup'.
    Adds a column 'Partner' with the partner name (e.g., 'CDC', 'NIH', etc) inferred from the folder name.
    Each CSV is expected to have the same structure.
    """
    exclude_folders = {'automation scripts', 'Logs', 'LogsBackup'}
    all_dfs = []
    today_str = datetime.today().strftime("%Y%m%d")
    for dirpath, dirnames, filenames in os.walk(root_path):
        # Exclude directories if any part of the path matches excluded names
        if any(ex in os.path.normpath(dirpath).split(os.sep) for ex in exclude_folders):
            continue
        partner = os.path.basename(dirpath)
        for fname in filenames:
            if fname.lower().endswith(f"{today_str}.csv"):
                fpath = os.path.join(dirpath, fname)
                try:
                    df = pd.read_csv(fpath)
                    df['Partner'] = partner
                    all_dfs.append(df)
                except Exception as e:
                    print(f"Skipping {fpath}: {e}")
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        print("No CSV files found for today.")
        return pd.DataFrame()

daily_search = load_all_csvs_from_folders(data_path_specific)
    
display(daily_search)

In [4]:
from datetime import datetime

save_path = r'Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Full Daily Reports'

today_str = datetime.today().strftime("%Y%m%d")
output_path = os.path.join(save_path, f"full_daily_report_{today_str}.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
daily_search.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Full Daily Reports\full_daily_report_20250623.csv


import os
import pandas as pd
from datetime import datetime
import re

data_path_specific = r'Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions'

def load_all_csvs_from_folders(root_path):
    """
    Recursively loads all CSV files from all subfolders under root_path into a single DataFrame.
    Excludes folders with titles 'automation scripts', 'Logs', or 'LogsBackup'.
    Adds columns 'Partner' (from folder name) and 'FileDate' (from filename, if present in YYYYMMDD format).
    Each CSV is expected to have the same structure.
    """
    exclude_folders = {'automation scripts', 'Logs', 'LogsBackup'}
    all_dfs = []
    date_pattern = re.compile(r'(\d{8})\.csv$', re.IGNORECASE)
    for dirpath, dirnames, filenames in os.walk(root_path):
        if any(ex in os.path.normpath(dirpath).split(os.sep) for ex in exclude_folders):
            continue
        partner = os.path.basename(dirpath)
        for fname in filenames:
            match = date_pattern.search(fname)
            if match:
                file_date = match.group(1)
                fpath = os.path.join(dirpath, fname)
                try:
                    df = pd.read_csv(fpath)
                    df['Partner'] = partner
                    df['FileDate'] = file_date
                    all_dfs.append(df)
                except Exception as e:
                    print(f"Skipping {fpath}: {e}")
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        print("No CSV files found.")
        return pd.DataFrame()

daily_search = load_all_csvs_from_folders(data_path_specific)
    
display(daily_search)

from datetime import datetime

save_path = r'Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Full Daily Reports'

for file_date in daily_search['FileDate'].unique():
    df_subset = daily_search[daily_search['FileDate'] == file_date]
    output_path = os.path.join(save_path, f"full_daily_report_{file_date}.csv")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_subset.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")
